# 8. Bonus: a GPU, and a model with no likelihood

Everything else in this project runs on a laptop CPU in seconds, which is what
makes it possible to iterate. This notebook is for the two questions small models
cannot answer.

**(a) Scale-up.** Convolutional versions of the methods on Track C, evaluated under
increasing corruption. The question is not which method wins but whether the
*ordering* you found with MLPs survives a real architecture.

**(b) Score matching.** A model that never represents a density at all, only its
gradient, plus annealed Langevin sampling to draw from it. Then the question of
what "uncertainty" means without an explicit likelihood.

Part (b) runs on a CPU in about ten seconds. Part (a) wants a GPU: set
`SMOKE = True` to check the whole code path on a laptop first, and comment out the
`[tool.uv.sources]` block in `pyproject.toml` before installing on Colab, because
it pins CPU-only wheels.

About 4 hours. Either part on its own is enough.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from bdl.data import corrupt, load_track
from bdl.metrics import accuracy, auroc, predictive_nll_probs, results_table
from bdl.models import fit, set_seed
from bdl.plots import plot_shift_sweep
from bdl.store import run_dir

%matplotlib inline

SMOKE = True  # True: tiny settings, runs on a CPU. False: the real thing, wants a GPU.


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = pick_device()
print("device:", device, " smoke:", SMOKE)

## 8.1 A convolutional network

Provided. It takes flattened images and reshapes internally, so it is a drop-in
replacement for the MLP everywhere else in the project.

In [ ]:
class SmallCNN(nn.Module):
    """A small convolutional network for Track C at full resolution."""

    def __init__(self, n_out, image_shape=(3, 28, 28), width=32, dropout=0.0):
        super().__init__()
        c, h, w = image_shape
        self.image_shape = image_shape
        self.features = nn.Sequential(
            nn.Conv2d(c, width, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(dropout),
            nn.Conv2d(width, 2 * width, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * width * (h // 4) * (w // 4), 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_out),
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x.reshape(-1, *self.image_shape)
        return self.head(self.features(x))


# The two functions you wrote in notebook 02, for the classification case.
def decompose_entropy(probs):
    eps = 1e-12
    probs = probs.clamp_min(eps)
    mean_p = probs.mean(dim=0)
    total = -(mean_p * torch.log(mean_p.clamp_min(eps))).sum(-1)
    aleatoric = -(probs * torch.log(probs)).sum(-1).mean(dim=0)
    return total, aleatoric, total - aleatoric


def calibration_error_probs(probs, y, n_bins=15):
    p = probs.mean(0)
    conf, hat = p.max(dim=-1)
    correct = (hat == y.reshape(-1)).float()
    edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if bool(in_bin.any()):
            ece += float(in_bin.float().mean()) * abs(
                float(correct[in_bin].mean()) - float(conf[in_bin].mean())
            )
    return ece

## 8.2 Part (a): does the ordering survive?

Train MC Dropout and a deep ensemble as CNNs, and evaluate both as the corruption
grows. What to look for in the report: the ranking against the MLP results from
notebook 05, and, where it changes, a reason. Candidates worth checking rather than
asserting: a convolutional prior changes what the same weight-space prior means in
function space; better features leave less for the last layer to be uncertain about;
an ensemble may find more distinct modes when the model is larger.

A third method to add if you want a wider comparison is last-layer Laplace: train
one network, then fit a Gaussian to the last layer using the curvature of the loss.
Section 8.4 sketches it.

In [ ]:
ds_c = load_track("C")
image_shape = (3, 28, 28) if ds_c.n_features == 2352 else (1, 28, 28)
epochs = 2 if SMOKE else 20
n_members = 2 if SMOKE else 5
n_samples = 5 if SMOKE else 20
strengths = [0.0, 0.5] if SMOKE else [0.0, 0.5, 1.0, 2.0]
print(ds_c, image_shape)


def train_cnn(seed, dropout):
    set_seed(seed)
    model = SmallCNN(ds_c.n_outputs, image_shape=image_shape, dropout=dropout).to(device)
    fit(
        model,
        ds_c.x_train,
        ds_c.y_train,
        loss_fn=lambda m, xb, yb, n: F.cross_entropy(m(xb), yb.reshape(-1)),
        epochs=epochs,
        lr=1e-3,
        batch_size=128,
        seed=seed,
        device=device,
    )
    return model


@torch.no_grad()
def predict_cnn(models, x, n_samples, mc_dropout):
    """Stacked softmax vectors, [S, N, K]. S = n_samples for dropout, len(models) otherwise."""
    out = []
    for model in models:
        model.eval()
        if mc_dropout:
            for m in model.modules():
                if isinstance(m, nn.Dropout):
                    m.train()
            for _ in range(n_samples):
                out.append(torch.softmax(model(x.to(device)), -1).cpu())
        else:
            out.append(torch.softmax(model(x.to(device)), -1).cpu())
    return torch.stack(out)


t0 = time.perf_counter()
methods = {
    "cnn_mc_dropout": ([train_cnn(0, dropout=0.2)], True),
    "cnn_ensemble": ([train_cnn(s, dropout=0.0) for s in range(n_members)], False),
}
print(f"trained in {time.perf_counter() - t0:.0f}s on {device}")

In [ ]:
results, sweep = {}, {}

for name, (models, mc) in methods.items():
    probs_id = predict_cnn(models, ds_c.x_test, n_samples, mc)
    probs_ood = predict_cnn(models, ds_c.x_ood, n_samples, mc)
    _, _, epi_id = decompose_entropy(probs_id)
    _, _, epi_ood = decompose_entropy(probs_ood)

    results[name] = {
        "nll": predictive_nll_probs(probs_id, ds_c.y_test),
        "accuracy": accuracy(probs_id, ds_c.y_test),
        "ece": calibration_error_probs(probs_id, ds_c.y_test),
        "epistemic_id": float(epi_id.mean()),
        "epistemic_ood": float(epi_ood.mean()),
        "ood_auroc": auroc(epi_id, epi_ood),
    }

    series = {"nll": [], "ece": [], "ood_auroc": []}
    for s in strengths:
        probs_s = predict_cnn(models, corrupt(ds_c.x_test, s, seed=1), n_samples, mc)
        _, _, epi_s = decompose_entropy(probs_s)
        series["nll"].append(predictive_nll_probs(probs_s, ds_c.y_test))
        series["ece"].append(calibration_error_probs(probs_s, ds_c.y_test))
        series["ood_auroc"].append(auroc(epi_id, epi_s))
    sweep[name] = series

print(results_table(results))
if len(strengths) > 1:
    plot_shift_sweep(sweep, strengths, run_dir("bonus_gpu") / "cnn_shift_sweep.png")

## 8.3 Part (b): learning a density by learning only its gradient

The score of a density is `s(x) = grad log p(x)`. It determines `p` up to
normalisation and never requires the normalising constant, which is what makes this
family possible at all.

Fitting the score to data looks circular, since the target `grad log p_data` is
unknown. Denoising score matching removes the circularity. Perturb the data with a
known kernel, `x_tilde = x + sigma * eps`, and fit the score of the *perturbed*
density. Up to a constant independent of the parameters,

    E || s(x_tilde) - grad log q(x_tilde) ||^2
      =  E || s(x_tilde) - grad log q(x_tilde | x) ||^2

and for a Gaussian kernel the right-hand target is known exactly:

    grad log q(x_tilde | x) = -(x_tilde - x) / sigma^2 = -eps / sigma

Weighting each noise level by `sigma^2` puts every scale on a comparable footing.
Without it the small-`sigma` terms, whose targets are enormous, dominate the loss:

    L = E || sigma * s(x_tilde, sigma) + eps ||^2

Note what this is: denoising. The network is never shown a density, never computes
a partition function, and what it learns is the gradient of the log-density.

In [ ]:
class ScoreNet(nn.Module):
    """A noise-conditioned score network s(x, sigma) for 2-D data.

    The noise level enters through a random Fourier embedding, the standard way to
    condition a network on a continuous scalar: feeding sigma in raw works badly
    because the network has to learn its many orders of magnitude itself.
    """

    def __init__(self, dim=2, hidden=128, n_freq=16):
        super().__init__()
        self.register_buffer("freqs", torch.randn(n_freq) * 2.0)
        self.net = nn.Sequential(
            nn.Linear(dim + 2 * n_freq, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, sigma):
        ang = torch.log(sigma).reshape(-1, 1) * self.freqs.reshape(1, -1)
        emb = torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)
        return self.net(torch.cat([x, emb], dim=-1))


def two_moons(n, seed=0):
    from sklearn.datasets import make_moons

    x, _ = make_moons(n_samples=n, noise=0.05, random_state=seed)
    return torch.tensor(x, dtype=torch.float32)

In [ ]:
def dsm_loss(model, x, sigmas):
    """Denoising score matching loss, averaged over a random noise level per example."""
    # ---- TODO ------------------------------------------------------------
    # draw one sigma per example from `sigmas`, perturb x_tilde = x + sigma * eps,
    # and return the mean of || sigma * model(x_tilde, sigma) + eps ||^2
    raise NotImplementedError
    # ----------------------------------------------------------------------


@torch.no_grad()
def annealed_langevin(model, n_samples, sigmas, n_steps=100, eps=2e-5, device=torch.device("cpu")):
    """Annealed Langevin dynamics, from the largest noise level down to the smallest."""
    # ---- TODO ------------------------------------------------------------
    # start from noise scaled by sigmas[0], then for each sigma in order run
    # n_steps updates of
    #     alpha = eps * (sigma / sigmas[-1]) ** 2
    #     x <- x + 0.5 * alpha * score(x, sigma) + sqrt(alpha) * z,  z ~ N(0, I)
    # The step size scales with sigma^2 so early levels move far and late levels
    # only polish. Drop the noise term and this becomes gradient ascent on the
    # log-density: every sample collapses onto the nearest mode. Worth trying once.
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work -------------------------------------------------------
torch.manual_seed(0)
sigmas_check = torch.exp(torch.linspace(np.log(1.0), np.log(0.01), 10))
net_check = ScoreNet()
x_check = two_moons(256)

loss = dsm_loss(net_check, x_check, sigmas_check)
assert loss.ndim == 0 and float(loss.detach()) > 0, loss
assert loss.requires_grad, "the loss must be differentiable"

# At the optimum the score of a Gaussian perturbation is known, so a network that
# is handed the right answer should score near zero. Check the loss really is
# minimised by the true score of the kernel rather than by something else.
class ExactScore(nn.Module):
    def __init__(self, data):
        super().__init__()
        self.data = data

    def forward(self, x_tilde, sigma):
        # For a single data point the exact score of the kernel is -(x_tilde - x)/sigma^2.
        return -(x_tilde - self.data) / sigma.reshape(-1, 1) ** 2


torch.manual_seed(0)
exact = ExactScore(x_check)
loss_exact = float(dsm_loss(exact, x_check, sigmas_check).detach())
torch.manual_seed(0)
loss_net = float(dsm_loss(net_check, x_check, sigmas_check).detach())
assert loss_exact < loss_net, f"exact score {loss_exact:.3f} should beat an untrained net {loss_net:.3f}"

samples = annealed_langevin(net_check, 16, sigmas_check, n_steps=5, device=torch.device("cpu"))
assert samples.shape == (16, 2), samples.shape
assert torch.isfinite(samples).all(), "the sampler produced non-finite values"
print(f"OK   loss with the exact kernel score {loss_exact:.4f} < untrained network {loss_net:.4f}")

In [ ]:
n_sigmas = 10
sigmas = torch.exp(torch.linspace(np.log(1.0), np.log(0.01), n_sigmas)).to(device)
x_data = two_moons(2048).to(device)

set_seed(0)
score_net = ScoreNet().to(device)
opt = torch.optim.Adam(score_net.parameters(), lr=2e-3)

steps = 3000  # about 10 s on a CPU; this part does not need a GPU
t0 = time.perf_counter()
for step in range(steps):
    opt.zero_grad(set_to_none=True)
    loss = dsm_loss(score_net, x_data, sigmas)
    loss.backward()
    opt.step()
    if step % max(steps // 5, 1) == 0:
        print(f"  step {step:5d}  loss {float(loss):.4f}")
print(f"trained in {time.perf_counter() - t0:.0f}s")

samples = annealed_langevin(score_net, 1000, sigmas, n_steps=100, device=device)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
d = x_data.cpu().numpy()
axes[0].plot(d[:, 0], d[:, 1], ".", ms=3, alpha=0.5)
axes[0].set_title("data")
s = samples.numpy()
axes[1].plot(s[:, 0], s[:, 1], ".", ms=3, alpha=0.5, color="#f0a202")
axes[1].set_title("annealed Langevin samples")

# The learned score field at the finest noise level.
gx, gy = np.meshgrid(np.linspace(-2, 3, 20), np.linspace(-1.5, 2, 20))
pts = torch.tensor(np.stack([gx.ravel(), gy.ravel()], -1), dtype=torch.float32, device=device)
with torch.no_grad():
    field = score_net(pts, sigmas[-1].expand(len(pts))).cpu().numpy()
axes[2].quiver(gx.ravel(), gy.ravel(), field[:, 0], field[:, 1], alpha=0.7)
axes[2].set_title("learned score field at the smallest sigma")
for ax in axes:
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1.5, 2)
fig.tight_layout()
fig.savefig(run_dir("bonus_gpu") / "score_matching.png", dpi=150, bbox_inches="tight")

## 8.4 Optional: last-layer Laplace, for a wider comparison in part (a)

Train an ordinary network to its MAP estimate, then fit a Gaussian to the posterior
around that point using the curvature of the loss:

    p(W | D) ~= N(W*, H^-1),   H = sum of per-example Hessians + prior precision

`H` is far too large to form, so three standard simplifications: use the diagonal
of the Fisher instead of the Hessian, which is a sum of squared per-example
gradients and is positive by construction; keep only the diagonal; and be Bayesian
about the last layer only, freezing the features. The recipe in code is

    for each training example:
        accumulate grad(loss)**2 for the last layer's parameters
    posterior_std = 1 / sqrt(fisher + prior_precision)

then sample the last layer's weights around the MAP values at prediction time and
restore them afterwards. Its attraction is cost: no retraining, one extra pass over
the training set. Its weakness is that a Gaussian bump at one mode is a poor
description of a multi-modal posterior, which is exactly what an ensemble exploits.

## 8.5 What "uncertainty" means without a likelihood

The discussion the assignment asks for. Most of this project's machinery does not
apply to a score model: there is no normalised density, so no NLL, no calibration in
the sense of notebook 01, and no proper scoring rule. What remains:

* relative densities along a path, from `log p(x1) - log p(x0) = integral of s . dx`.
  That is path-independent only if `s` is a true gradient field, which an
  unconstrained network is not, so measuring the path dependence is itself a
  diagnostic;
* the norm of the score as an out-of-distribution signal, with the caveat that it is
  large near sharp features of the density and not only far from the data;
* an exact likelihood via the probability-flow ODE, which turns the score model into
  a continuous normalizing flow and buys back `log p` at the cost of an ODE solve.
  That is how diffusion models report likelihoods.

## What to report

* which parts ran on a GPU, at what scale, and how long they took;
* for part (a): the ranking under corruption next to your MLP ranking from notebook
  05, and a reason for any change;
* for part (b): the three-panel figure, and what happens when you drop the noise
  term from the Langevin update.